# 02 · Select, filter, and sort survey observations

**Module:** 2 · **Time:** 40 minutes · **Prerequisites:** Module 2: DataFrames introduction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/02_selecting_filtering.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F02_selecting_filtering.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is currently private and these badges refer to `main` after merge.

## Learning goals

- **S**elect research variables and observations using loc and iloc.
- **F**ilter and count survey groups with Boolean conditions.
- **S**ort observations while avoiding chained assignment.

In [ ]:
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print(survey.head(2))

## Why this matters

Empirical researchers constantly define analysis samples: employed respondents, a subset of geographic regions, or records with usable weights. Selecting the wrong observations can change every result. This lesson uses a six-row fictional practice survey, with no real people or inferential claims.

## Select columns and positions

A single bracketed column returns a Series; a list of column names returns a DataFrame. `.loc` selects by labels and `.iloc` by integer positions.

In [ ]:
selected = survey[["person_id", "region", "survey_weight"]]
first_two = survey.iloc[:2, :3]
print(selected.head(2))
print(first_two)

## Filter with loc and Boolean conditions

A comparison produces a Boolean mask. Use `.loc[rows, columns]` for readable, stable selection and updates.

In [ ]:
is_employed = survey["employment"].eq("employed")
employed = survey.loc[is_employed, ["person_id", "region"]]
print(employed)
print("Selected rows:", len(employed))

## Use query, isin, and value counts

`query` expresses simple conditions; `isin` keeps any of several values. When text has inconsistent capitalization or spaces, standardize it first.

In [ ]:
north_or_south = survey["region"].str.strip().str.lower().isin(["north", "south"])
regional = survey.loc[north_or_south].copy()
print(regional["employment"].value_counts())
print(survey.query("survey_weight >= 1.0")[["person_id", "survey_weight"]])

## Sort and assign safely under Copy-on-Write

pandas 3 uses Copy-on-Write: chained assignment cannot reliably update its original DataFrame. Set values directly with `.loc` on the object you intend to modify.

In [ ]:
ordered = survey.sort_values("survey_weight", ascending=False)
safe = survey.copy()
safe.loc[safe["employment"].eq("employed"), "eligible"] = True
safe["eligible"] = safe["eligible"].fillna(False).astype(bool)
print(ordered[["person_id", "survey_weight"]].head(3))
print(safe["eligible"].value_counts())

## If you know Stata/R

| Task | Python / pandas | Stata | R |
| --- | --- | --- | --- |
| Keep columns | `df[["id", "wage"]]` | `keep id wage` | `dplyr::select(df, id, wage)` |
| Filter rows | `df.loc[df["wage"] > 0]` | `keep if wage > 0` | `dplyr::filter(df, wage > 0)` |
| Test membership | `df["region"].isin(values)` | `inlist(region, ...)` | `region %in% values` |
| Sort | `df.sort_values("wage")` | `sort wage` | `dplyr::arrange(df, wage)` |

`.loc` assignments are recommended for pandas 2.x and 3.x. pandas 3's Copy-on-Write behavior makes chained assignment especially misleading.

## Try it

Complete each blank answer cell, then run its friendly check. In the learner version, checks print hints instead of raising an exception.

### ★ Exercise 1 — Keep three variables

Create a DataFrame `core_columns` containing `person_id`, `employment`, and `survey_weight`, in that order.

In [ ]:
core_columns = None

In [ ]:
expected = survey[["person_id", "employment", "survey_weight"]]
if isinstance(core_columns, pd.DataFrame) and core_columns.equals(expected):
    print("✓ Columns selected")
else:
    print("Hint: pass a list of column names inside double brackets.")

### ★★ Exercise 2 — Filter employed people

Create `employed_rows` with all rows whose `employment` equals `employed`.

In [ ]:
employed_rows = None

In [ ]:
expected = survey.loc[survey["employment"].eq("employed")]
if isinstance(employed_rows, pd.DataFrame) and employed_rows.equals(expected):
    print("✓ Employment sample correct")
else:
    print("Hint: use a Boolean mask and .loc to select rows.")

### ★★ Exercise 3 — Find North across inconsistent spelling

Create `north_rows` where `region` equals `north` after stripping spaces and lowercasing.

In [ ]:
north_rows = None

In [ ]:
expected = survey.loc[survey["region"].str.strip().str.lower().eq("north")]
if isinstance(north_rows, pd.DataFrame) and north_rows.equals(expected):
    print("✓ North rows correct")
else:
    print("Hint: combine .str.strip(), .str.lower(), and .eq('north').")

### ★★★ Exercise 4 — Sort the top three weights

Set `top_weights` to the three rows with the largest `survey_weight`, sorted descending.

In [ ]:
top_weights = None

In [ ]:
expected = survey.sort_values("survey_weight", ascending=False).head(3)
if isinstance(top_weights, pd.DataFrame) and top_weights.equals(expected):
    print("✓ Top weights correct")
else:
    print("Hint: sort_values(..., ascending=False), then head(3).")

## Common errors

- `KeyError` — the requested column name does not match; print `survey.columns`.
- Chained assignment such as `df['x'][mask] = 1` is not a safe way to modify a DataFrame; prefer `df.loc[mask, 'x'] = 1`.
- Comparisons on messy text miss cases such as ` North ` versus `north`; normalize text before filtering.
- `.iloc` uses integer positions while `.loc` uses labels; confusing them selects unintended observations.

## Recap

- Select columns with a list and rows with Boolean masks.
- Use `query`, `isin`, and `value_counts` for readable sample summaries.
- Prefer direct `.loc` assignments, particularly with pandas 3 Copy-on-Write.
- Check the sample size after every filter.

## Next steps

Next: [Cleaning a messy survey](03_cleaning.ipynb). See the [pandas Copy-on-Write guide](https://pandas.pydata.org/docs/user_guide/copy_on_write.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) and include the lesson path.